In [ ]:
Problem Statement : Hospital Patient Data Analysis
Context:
A hospital maintains patient records including admission details, department, diagnosis, doctor, and bill amount. You have two datasets: one with patient info and another with billing details. Some patients have blank bill amounts, and there are multiple rows for the same patient due to follow-ups.


In [3]:
import pandas as pd

patient_df = pd.read_csv("/Users/SS/Downloads/Patient_Data.csv")
patient_df


,PatientID,Name,Department,Doctor,BillAmount,ReceptionistID,CheckInTime
0,101,Alice,Cardiology,Dr. Smith,5000.0,1,2023-01-10 09:00
1,102,Bob,Neurology,Dr. John,NaN,2,2023-01-11 10:30
2,103,Charlie,Orthopedics,Dr. Lee,7500.0,1,2023-01-12 11:00
3,104,David,Cardiology,Dr. Smith,6200.0,3,2023-01-13 12:00
4,105,Eva,Dermatology,Dr. Rose,NaN,2,2023-01-14 08:45
5,101,Alice,Cardiology,Dr. Smith,5000.0,1,2023-01-10 09:00


In [6]:
billing_df = pd.read_csv("/Users/SS/Downloads/Billing_Data.csv")
billing_df

,PatientID,InsuranceCovered,FinalAmount
0,101,2000,3000
1,102,1500,3500
2,103,2500,5000
3,104,3000,3200
4,105,1000,4000


In [ ]:
#1.	Load the patient dataset and show summary with info().

In [7]:
print("Patient Dataset Info:")
print(patient_df.info())

Patient Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   PatientID       6 non-null      int64  
 1   Name            6 non-null      object 
 2   Department      6 non-null      object 
 3   Doctor          6 non-null      object 
 4   BillAmount      4 non-null      float64
 5   ReceptionistID  6 non-null      int64  
 6   CheckInTime     6 non-null      object 
dtypes: float64(1), int64(2), object(4)
memory usage: 468.0+ bytes
None


In [ ]:
#2.	Select only the columns relevant for billing: ['PatientID', 'Department', 'Doctor', 'BillAmount'].

In [8]:
cols = ['PatientID', 'Department', 'Doctor', 'BillAmount']

patient_df.columns = patient_df.columns.str.strip()  # remove spaces
billing_df.columns = billing_df.columns.str.strip()

selected_df = patient_df[cols]
print("\nSelected Columns:")
print(selected_df.head())


Selected Columns:
   PatientID   Department     Doctor  BillAmount
0        101   Cardiology  Dr. Smith      5000.0
1        102    Neurology   Dr. John         NaN
2        103  Orthopedics    Dr. Lee      7500.0
3        104   Cardiology  Dr. Smith      6200.0
4        105  Dermatology   Dr. Rose         NaN


In [ ]:
#3.	Drop administrative columns like ['ReceptionistID', 'CheckInTime'].

In [13]:
patient_df = patient_df.drop(columns=['ReceptionistID', 'CheckInTime'], errors='ignore')

print(patient_df.head())

   PatientID     Name   Department     Doctor   BillAmount
0        101    Alice   Cardiology  Dr. Smith  5000.000000
1        102      Bob    Neurology   Dr. John  6233.333333
2        103  Charlie  Orthopedics    Dr. Lee  7500.000000
3        104    David   Cardiology  Dr. Smith  6200.000000
4        105      Eva  Dermatology   Dr. Rose  6233.333333


In [ ]:
#4.	Use groupby to find total bill amount per department.

In [10]:
dept_bill = selected_df.groupby('Department')['BillAmount'].sum()
print("\nTotal Bill per Department:")
print(dept_bill)


Total Bill per Department:
Department
Cardiology     16200.0
Dermatology        0.0
Neurology          0.0
Orthopedics     7500.0
Name: BillAmount, dtype: float64


In [ ]:
#5.	Remove duplicate patient records based on PatientID.

In [14]:
patient_df = patient_df.drop_duplicates(subset='PatientID')

print(patient_df.head())

   PatientID     Name   Department     Doctor   BillAmount
0        101    Alice   Cardiology  Dr. Smith  5000.000000
1        102      Bob    Neurology   Dr. John  6233.333333
2        103  Charlie  Orthopedics    Dr. Lee  7500.000000
3        104    David   Cardiology  Dr. Smith  6200.000000
4        105      Eva  Dermatology   Dr. Rose  6233.333333


In [ ]:
#6.	Fill missing BillAmount values with the mean bill amount.

In [15]:
mean_bill = patient_df['BillAmount'].mean()

patient_df['BillAmount'] = patient_df['BillAmount'].fillna(mean_bill)

print(patient_df.head())

   PatientID     Name   Department     Doctor   BillAmount
0        101    Alice   Cardiology  Dr. Smith  5000.000000
1        102      Bob    Neurology   Dr. John  6233.333333
2        103  Charlie  Orthopedics    Dr. Lee  7500.000000
3        104    David   Cardiology  Dr. Smith  6200.000000
4        105      Eva  Dermatology   Dr. Rose  6233.333333


In [ ]:
7.	Merge the billing dataset with patient dataset on PatientID.

In [16]:
billing_df.columns = billing_df.columns.str.strip()

merged_df = pd.merge(patient_df, billing_df, on='PatientID', how='inner')

print(merged_df.head())


   PatientID     Name   Department     Doctor   BillAmount  InsuranceCovered  \
0        101    Alice   Cardiology  Dr. Smith  5000.000000              2000   
1        102      Bob    Neurology   Dr. John  6233.333333              1500   
2        103  Charlie  Orthopedics    Dr. Lee  7500.000000              2500   
3        104    David   Cardiology  Dr. Smith  6200.000000              3000   
4        105      Eva  Dermatology   Dr. Rose  6233.333333              1000   

   FinalAmount  
0         3000  
1         3500  
2         5000  
3         3200  
4         4000  


In [ ]:
8.	Concatenate an additional DataFrame that contains new patients for the current week (row-wise).

In [17]:
new_patients = pd.DataFrame({
    'PatientID': [101, 102],
    'Department': ['Cardiology', 'Neurology'],
    'Doctor': ['Dr.X', 'Dr.Y'],
    'BillAmount': [5000, 7000]
})

final_df = pd.concat([merged_df, new_patients], ignore_index=True)

print(final_df.tail())

   PatientID     Name   Department     Doctor   BillAmount  InsuranceCovered  \
2        103  Charlie  Orthopedics    Dr. Lee  7500.000000            2500.0   
3        104    David   Cardiology  Dr. Smith  6200.000000            3000.0   
4        105      Eva  Dermatology   Dr. Rose  6233.333333            1000.0   
5        101      NaN   Cardiology       Dr.X  5000.000000               NaN   
6        102      NaN    Neurology       Dr.Y  7000.000000               NaN   

   FinalAmount  
2       5000.0  
3       3200.0  
4       4000.0  
5          NaN  
6          NaN  


In [ ]:
9.	Concatenate new billing category columns like ['InsuranceCovered', 'FinalAmount'] (column-wise).

In [18]:
final_df['InsuranceCovered'] = True
final_df['FinalAmount'] = final_df['BillAmount'] * 0.9

print(final_df.head())

   PatientID     Name   Department     Doctor   BillAmount  InsuranceCovered  \
0        101    Alice   Cardiology  Dr. Smith  5000.000000              True   
1        102      Bob    Neurology   Dr. John  6233.333333              True   
2        103  Charlie  Orthopedics    Dr. Lee  7500.000000              True   
3        104    David   Cardiology  Dr. Smith  6200.000000              True   
4        105      Eva  Dermatology   Dr. Rose  6233.333333              True   

   FinalAmount  
0       4500.0  
1       5610.0  
2       6750.0  
3       5580.0  
4       5610.0  


In [19]:
print("\nFinal Cleaned Dataset:")
print(final_df.head())

print("\nDataset Info:")
print(final_df.info())


Final Cleaned Dataset:
   PatientID     Name   Department     Doctor   BillAmount  InsuranceCovered  \
0        101    Alice   Cardiology  Dr. Smith  5000.000000              True   
1        102      Bob    Neurology   Dr. John  6233.333333              True   
2        103  Charlie  Orthopedics    Dr. Lee  7500.000000              True   
3        104    David   Cardiology  Dr. Smith  6200.000000              True   
4        105      Eva  Dermatology   Dr. Rose  6233.333333              True   

   FinalAmount  
0       4500.0  
1       5610.0  
2       6750.0  
3       5580.0  
4       5610.0  

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   PatientID         7 non-null      int64  
 1   Name              5 non-null      object 
 2   Department        7 non-null      object 
 3   Doctor            7 non-null      obje